In [ ]:
# Configuração para Google Colab (instalação automática de dependências extras)
import sys
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Instalando pacotes adicionais no Google Colab...")
    # ultralytics (YOLO) e easyocr não vêm instalados por padrão
    get_ipython().system('pip install -q ultralytics easyocr pytesseract')
    get_ipython().system('apt-get install -q -y tesseract-ocr')
    print("Tudo pronto!")

# TCC Experimento 2: Reconhecimento de Placas (UFPR-ALPR)

**Objetivo**: Avaliar a precisão do pipeline ALPR completo (Detecção de Veículo + Detecção de Placa + OCR).

---
**Métricas**: IoU (Placa), CER (OCR), Exact Match (Acurácia).

## 1. Configuração e Ambiente

In [ ]:
import sys, cv2, time, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import display
import torch

BASE_DIR = Path.cwd()
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
%matplotlib inline

UFPR_ROOT      = BASE_DIR / "UFPR-ALPR dataset"
RESULTS_DIR    = BASE_DIR / "dataset_processado" / "resultados"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# LIMITAÇÃO DE EXECUÇÃO: Ajuste aqui para testes rápidos
LIMIT = 50  # Defina como None para rodar o dataset completo (1500 imagens)

print(f"UFPR-ALPR dataset: {'Encontrado' if UFPR_ROOT.exists() else 'NÃO ENCONTRADO'}")
print(f"Dispositivo GPU disponível: {torch.cuda.is_available()}")

## 2. Análise e Visualização do Dataset UFPR-ALPR

In [ ]:
# =============================================================================
# CÓDIGO AUXILIAR (DETECTORES, OCR, PARSER, AVALIAÇÃO E BENCHMARK) - CONSOLIDADO
# =============================================================================
import torch
import cv2
import numpy as np
import pandas as pd
import time
import random
import re
import easyocr
import pytesseract
from pathlib import Path
from abc import ABC, abstractmethod
from ultralytics import YOLO
from torchvision.models.detection import fasterrcnn_resnet50_fpn, ssdlite320_mobilenet_v3_large
from torchvision.models.detection import FasterRCNN_ResNet50_FPN_Weights, SSDLite320_MobileNet_V3_Large_Weights
from tqdm import tqdm

# --- PARSER UFPR-ALPR ---
from pathlib import Path


def parse_annotation(txt_path: Path) -> dict:
    """
    Lê um arquivo .txt do UFPR-ALPR e retorna um dicionário com:
        - plate_text   : str   — ex: 'AQY6388'
        - vehicle_bbox : list  — [x1, y1, x2, y2]
        - plate_bbox   : list  — [x1, y1, x2, y2] (bounding rect dos 4 cantos)
        - plate_corners: list  — [[x,y], [x,y], [x,y], [x,y]]
    Retorna None se o arquivo não puder ser lido.
    """
    try:
        data = {}
        with open(txt_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()

        for line in lines:
            line = line.strip()

            # Posição do veículo: x y w h
            if line.startswith('position_vehicle:'):
                parts = line.split(':')[1].strip().split()
                x, y, w, h = int(parts[0]), int(parts[1]), int(parts[2]), int(parts[3])
                data['vehicle_bbox'] = [x, y, x + w, y + h]

            # Texto da placa
            elif line.startswith('plate:'):
                data['plate_text'] = line.split(':')[1].strip()

            # Cantos da placa: x1,y1 x2,y2 x3,y3 x4,y4
            elif line.startswith('corners:'):
                corners_str = line.split(':')[1].strip().split()
                corners = []
                for c in corners_str:
                    cx, cy = c.split(',')
                    corners.append([int(cx), int(cy)])
                data['plate_corners'] = corners

                # Calcula bounding box retangular a partir dos 4 cantos
                xs = [p[0] for p in corners]
                ys = [p[1] for p in corners]
                data['plate_bbox'] = [min(xs), min(ys), max(xs), max(ys)]

        return data if 'plate_text' in data else None

    except Exception as e:
        print(f"Erro ao parsear {txt_path}: {e}")
        return None


def load_dataset_split(dataset_root: Path, split: str = 'testing', limit: int = None) -> list:
    """
    Carrega todos os pares (imagem, anotação) de um split do UFPR-ALPR.

    Args:
        dataset_root: caminho para a pasta 'UFPR-ALPR dataset'
        split: 'testing', 'training' ou 'validation'
        limit: número máximo de amostras a carregar (None = todas)

    Returns:
        Lista de dicts com chaves: 'image_path', 'annotation'
    """
    split_path = dataset_root / split
    samples = []

    # O dataset é organizado em track0001/, track0002/, etc.
    for track_dir in sorted(split_path.iterdir()):
        if not track_dir.is_dir():
            continue
        for img_path in sorted(track_dir.glob('*.png')):
            txt_path = img_path.with_suffix('.txt')
            if not txt_path.exists():
                continue
            annotation = parse_annotation(txt_path)
            if annotation:
                samples.append({
                    'image_path': img_path,
                    'annotation': annotation
                })
            if limit and len(samples) >= limit:
                return samples

    return samples

# --- DETECTORES DE VEÍCULOS E PLACAS ---
class BaseDetector(ABC):
    @abstractmethod
    def detect(self, frame):
        """Retorna lista de detecções: [{'bbox': [x1, y1, x2, y2], 'conf': 0.9, 'class': 'car'}]"""
        pass

class YOLODetector(BaseDetector):
    def __init__(self, model_path='yolov8n.pt'):
        self.model = YOLO(model_path)
        self.model_name = "YOLOv8"
        
    def detect(self, frame):
        results = self.model(frame, verbose=False)[0]
        detections = []
        for box in results.boxes:
            detections.append({
                'bbox': box.xyxy[0].tolist(),
                'conf': float(box.conf),
                'class': self.model.names[int(box.cls)]
            })
        return detections

class TorchvisionDetector(BaseDetector):
    def __init__(self, model_type='faster_rcnn', confidence_threshold=0.5):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.threshold = confidence_threshold
        self.model_name = "Faster R-CNN" if model_type == 'faster_rcnn' else "SSD"
        
        if model_type == 'faster_rcnn':
            weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
            self.model = fasterrcnn_resnet50_fpn(weights=weights).to(self.device)
            self.classes = weights.meta["categories"]
        else: # SSD
            weights = SSDLite320_MobileNet_V3_Large_Weights.DEFAULT
            self.model = ssdlite320_mobilenet_v3_large(weights=weights).to(self.device)
            self.classes = weights.meta["categories"]
            
        self.model.eval()

    def detect(self, frame):
        # Preprocess
        img_tensor = torch.from_numpy(frame).permute(2, 0, 1).float().div(255).unsqueeze(0).to(self.device)
        
        with torch.no_grad():
            prediction = self.model(img_tensor)[0]
        
        detections = []
        for i in range(len(prediction['boxes'])):
            score = float(prediction['scores'][i])
            if score > self.threshold:
                detections.append({
                    'bbox': prediction['boxes'][i].tolist(),
                    'conf': score,
                    'class': self.classes[int(prediction['labels'][i])]
                })
        return detections

class PlateDetector(BaseDetector):
    def __init__(self, model_path='yolov8n-plate.pt'):
        try:
            # Tenta carregar o modelo YOLO especializado
            self.model = YOLO(model_path)
            self.has_model = True
        except Exception as e:
            print(f"Aviso: Modelo YOLO de placas não encontrado. Usando fallback OpenCV.")
            self.has_model = False
        self.model_name = "PlateDetector"
        
    def detect(self, frame):
        if self.has_model:
            results = self.model(frame, verbose=False)[0]
            detections = []
            for box in results.boxes:
                detections.append({
                    'bbox': box.xyxy[0].tolist(),
                    'conf': float(box.conf),
                    'class': 'plate'
                })
            return detections
        else:
            # --- FALLBACK: Visão Computacional Clássica ---
            # Ideal para TCC: Detecção baseada em bordas e contornos
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            # Filtro para reduzir ruído mantendo bordas
            bfilter = cv2.bilateralFilter(gray, 11, 17, 17)
            # Detecção de bordas
            edged = cv2.Canny(bfilter, 30, 200)
            
            # Encontrar contornos
            keypoints = cv2.findContours(edged.copy(), cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
            contours = sorted(keypoints[0], key=cv2.contourArea, reverse=True)[:10]
            
            detections = []
            for res in contours:
                approx = cv2.approxPolyDP(res, 10, True)
                if len(approx) == 4: # Retângulos (possíveis placas)
                    x, y, w, h = cv2.boundingRect(res)
                    aspect_ratio = w / float(h)
                    # Placas brasileiras têm proporção de ~3:1 a ~4:1
                    if 2.0 < aspect_ratio < 5.0:
                        detections.append({
                            'bbox': [float(x), float(y), float(x+w), float(y+h)],
                            'conf': 0.8, # Confiança fixa para o fallback
                            'class': 'plate'
                        })
            return detections

# --- OCR ENGINE ---

# ---------------------------------------------------------------------------
# Mapeamentos de correção por posição para placas brasileiras
#
# Formato antigo:  L L L N N N N   (ex: MLS5511)
# Formato Mercosul: L L L N L N N  (ex: ABC1D23)
#
# O OCR frequentemente troca:
#   letras por números: O↔0, I↔1, S↔5, B↔8, Z↔2, G↔6, Q↔0
#   números por letras: 0↔O, 1↔I, 5↔S, 8↔B, 2↔Z, 6↔G
# ---------------------------------------------------------------------------

# Caracteres que são letras mas parecem números
_NUM_TO_LETTER = str.maketrans('0158268479', 'OISBZGAHQ?')  # só se position for letra
# Caracteres que são números mas parecem letras
_LETTER_TO_NUM = str.maketrans('OISBZGAHQ', '015826649')   # só se position for dígito


def _fix_char(ch: str, expect_letter: bool) -> str:
    """Corrige um caractere OCR com base no tipo esperado na posição."""
    ch = ch.upper()
    if expect_letter:
        return ch.translate(_NUM_TO_LETTER)
    else:
        return ch.translate(_LETTER_TO_NUM)


def _is_mercosul(raw: str) -> bool:
    """Heurística para detectar se a placa está no formato Mercosul (AAA0A00)."""
    if len(raw) != 7:
        return False
    # Mercosul: posição 4 (índice 3) é número, posição 5 (índice 4) é letra
    # Antigo:   posições 4-7 (índices 3-6) são todos números
    return raw[4].isalpha() if raw[4].isascii() else False


def _apply_plate_mask(raw: str) -> str:
    """
    Aplica máscara de posição para corrigir confusões letra/número do OCR.

    Formato antigo:   L L L N N N N  (posições 0,1,2 = letra; 3,4,5,6 = dígito)
    Formato Mercosul: L L L N L N N  (posições 0,1,2 = letra; 3 = dígito;
                                       4 = letra; 5,6 = dígito)
    """
    if len(raw) < 7:
        return raw  # Muito curto — não tenta corrigir

    mercosul = _is_mercosul(raw)

    if mercosul:
        mask = [True, True, True, False, True, False, False]  # True = espera letra
    else:
        mask = [True, True, True, False, False, False, False]

    corrected = []
    for i, ch in enumerate(raw[:7]):
        if i < len(mask):
            corrected.append(_fix_char(ch, expect_letter=mask[i]))
        else:
            corrected.append(ch)

    return ''.join(corrected)


class OCREngine:
    def __init__(self, engine_type='easyocr'):
        self.engine_type = engine_type
        if engine_type == 'easyocr':
            # Português + inglês; GPU se disponível
            self.reader = easyocr.Reader(['pt', 'en'], gpu=True)

    # ------------------------------------------------------------------
    # Limpeza de texto bruto
    # ------------------------------------------------------------------
    def clean_text(self, text: str) -> str:
        """Remove tudo que não é letra ou dígito e converte para maiúsculo."""
        return re.sub(r'[^A-Z0-9]', '', text.upper())

    # ------------------------------------------------------------------
    # Pré-processamento da imagem da placa
    # ------------------------------------------------------------------
    def preprocess_plate(self, plate_crop: np.ndarray) -> np.ndarray:
        """
        Pipeline de pré-processamento otimizado para placas brasileiras.

        1. Upscaling 3× (melhora resolução para o OCR)
        2. Conversão para cinza
        3. CLAHE (equalização adaptativa de histograma — melhora contraste)
        4. Denoising leve (preserva bordas de caracteres)
        5. Threshold de Otsu (binarização global — mais estável que o adaptativo
           para placas com fundo uniforme)
        """
        h, w = plate_crop.shape[:2]
        # 1. Upscaling
        plate = cv2.resize(plate_crop, (w * 3, h * 3), interpolation=cv2.INTER_CUBIC)

        # 2. Cinza
        gray = cv2.cvtColor(plate, cv2.COLOR_BGR2GRAY)

        # 3. CLAHE — melhora contraste local sem destruir bordas finas
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(4, 4))
        gray = clahe.apply(gray)

        # 4. Denoising leve
        gray = cv2.fastNlMeansDenoising(gray, h=10, templateWindowSize=7, searchWindowSize=21)

        # 5. Threshold de Otsu
        _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

        return binary

    # ------------------------------------------------------------------
    # Leitura da placa
    # ------------------------------------------------------------------
    def read_plate(self, plate_crop: np.ndarray) -> str:
        """
        Lê o texto de uma imagem de placa recortada.

        Etapas:
          1. Pré-processamento da imagem
          2. OCR (EasyOCR ou Tesseract)
          3. Limpeza do texto bruto
          4. Correção posicional letra/número (máscara de placa brasileira)
        """
        if plate_crop is None or plate_crop.size == 0:
            return ""

        processed = self.preprocess_plate(plate_crop)

        raw = ""
        if self.engine_type == 'easyocr':
            results = self.reader.readtext(processed)
            if results:
                # Pega a leitura com maior confiança
                raw = max(results, key=lambda x: x[2])[1]
        else:
            config = (
                '--psm 7 '
                '-c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789'
            )
            raw = pytesseract.image_to_string(processed, config=config)

        cleaned = self.clean_text(raw)

        # Aplica correção posicional se o texto tiver comprimento de placa válido (7)
        if len(cleaned) == 7:
            cleaned = _apply_plate_mask(cleaned)

        return cleaned

# --- AVALIAÇÃO (IoU, CER) ---

def compute_iou(box_a: list, box_b: list) -> float:
    """
    Calcula o IoU entre duas bounding boxes no formato [x1, y1, x2, y2].

    IoU = Área de Interseção / Área de União

    Retorna um valor entre 0.0 (sem sobreposição) e 1.0 (sobreposição perfeita).
    """
    # Interseção
    x1 = max(box_a[0], box_b[0])
    y1 = max(box_a[1], box_b[1])
    x2 = min(box_a[2], box_b[2])
    y2 = min(box_a[3], box_b[3])

    inter_w = max(0, x2 - x1)
    inter_h = max(0, y2 - y1)
    inter_area = inter_w * inter_h

    if inter_area == 0:
        return 0.0

    # Área de cada caixa
    area_a = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
    area_b = (box_b[2] - box_b[0]) * (box_b[3] - box_b[1])

    union_area = area_a + area_b - inter_area
    return inter_area / union_area if union_area > 0 else 0.0


def compute_cer(predicted: str, ground_truth: str) -> float:
    """
    Calcula o CER (Character Error Rate) entre dois textos.

    CER = (Substituições + Inserções + Deleções) / len(ground_truth)

    Baseado na distância de edição de Levenshtein.
    Retorna 0.0 para acerto perfeito, 1.0 para erro total.
    """
    pred = predicted.upper().replace(' ', '') if predicted else ''
    gt   = ground_truth.upper().replace(' ', '') if ground_truth else ''

    if not gt:
        return 0.0 if not pred else 1.0

    # Matriz de Levenshtein
    n, m = len(gt), len(pred)
    dp = list(range(m + 1))

    for i in range(1, n + 1):
        prev = dp[:]
        dp[0] = i
        for j in range(1, m + 1):
            if gt[i - 1] == pred[j - 1]:
                dp[j] = prev[j - 1]
            else:
                dp[j] = 1 + min(prev[j], dp[j - 1], prev[j - 1])

    return min(dp[m] / n, 1.0)


def is_exact_match(predicted: str, ground_truth: str) -> bool:
    """Retorna True se a placa foi lida exatamente correta (ignorando maiúsculas)."""
    pred = predicted.upper().replace(' ', '').replace('-', '') if predicted else ''
    gt   = ground_truth.upper().replace(' ', '').replace('-', '') if ground_truth else ''
    return pred == gt


def evaluate_detection(predicted_bbox: list, ground_truth_bbox: list,
                        predicted_text: str, ground_truth_text: str) -> dict:
    """
    Avalia uma única detecção de placa contra o ground truth.

    Retorna dict com:
        - iou         : float (0-1)
        - cer         : float (0-1, menor é melhor)
        - exact_match : bool
        - plate_found : bool (IoU > 0.3)
    """
    iou = compute_iou(predicted_bbox, ground_truth_bbox) if predicted_bbox else 0.0
    cer = compute_cer(predicted_text, ground_truth_text)
    match = is_exact_match(predicted_text, ground_truth_text)

    return {
        'iou': iou,
        'cer': cer,
        'exact_match': match,
        'plate_found': iou > 0.3,  # Limiar padrão para "detectou corretamente"
    }

# --- RUN BENCHMARK ALPR ---
def run_benchmark(split: str = 'testing', limit: int = None):
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    print("=" * 60)
    print("  BENCHMARK ALPR — UFPR-ALPR Dataset (Ground Truth Real)")
    print("=" * 60)

    # --- Carrega amostras do split ---
    print(f"\nCarregando split '{split}'...")
    samples = load_dataset_split(DATASET_ROOT, split=split, limit=limit)
    print(f"  Total de amostras carregadas: {len(samples)}")

    # --- Detectores de veículo ---
    models_config = [
        ('YOLOv8',       lambda: YOLODetector('yolov8n.pt')),
        ('SSD',          lambda: TorchvisionDetector(model_type='ssd')),
        ('Faster R-CNN', lambda: TorchvisionDetector(model_type='faster_rcnn')),
    ]

    # Tenta inicializar cada modelo, pula se falhar
    vehicle_detectors = []
    for name, factory in models_config:
        try:
            print(f"  Carregando {name}...", end=' ')
            det = factory()
            vehicle_detectors.append((name, det))
            print("OK")
        except Exception as e:
            print(f"FALHOU ({type(e).__name__}). Pulando.")

    if not vehicle_detectors:
        print("Nenhum detector carregou com sucesso. Abortando.")
        return None

    # --- Detector de placa (único, compartilhado) ---
    plate_detector = PlateDetector()

    # --- OCR ---
    ocr = OCREngine(engine_type='easyocr')

    all_results = []

    for det_name, vehicle_det in vehicle_detectors:
        print(f"\n>>> Testando: {det_name}")

        for sample in tqdm(samples, desc=det_name):
            img_path   = sample['image_path']
            annotation = sample['annotation']
            gt_text    = annotation['plate_text']
            gt_bbox    = annotation['plate_bbox']       # ground truth da placa
            gt_veh     = annotation['vehicle_bbox']     # ground truth do veículo

            img = cv2.imread(str(img_path))
            if img is None:
                continue

            h_img, w_img = img.shape[:2]

            # --- Estágio 1: Usa o detector real para encontrar o veículo ---
            vehicle_detections = vehicle_det.detect(img)
            vehicles = [d for d in vehicle_detections
                        if d['class'] in ['car', 'truck', 'bus', 'motorcycle']]

            if vehicles:
                # Pega o maior veículo detectado
                v = max(vehicles, key=lambda d: (d['bbox'][2]-d['bbox'][0]) * (d['bbox'][3]-d['bbox'][1]))
                x1, y1, x2, y2 = map(int, v['bbox'])
                # Clipa para dentro da imagem
                x1, y1 = max(0, x1), max(0, y1)
                x2, y2 = min(w_img, x2), min(h_img, y2)
                vehicle_crop = img[y1:y2, x1:x2]
                # Ajusta gt_bbox para o sistema de coordenadas do recorte
                gt_bbox_crop = [
                    gt_bbox[0] - x1, gt_bbox[1] - y1,
                    gt_bbox[2] - x1, gt_bbox[3] - y1,
                ]
            else:
                # Fallback: usa veículo do ground truth se o detector falhar
                x1, y1 = max(0, gt_veh[0]), max(0, gt_veh[1])
                x2, y2 = min(w_img, gt_veh[2]), min(h_img, gt_veh[3])
                vehicle_crop = img[y1:y2, x1:x2]
                gt_bbox_crop = [
                    gt_bbox[0] - x1, gt_bbox[1] - y1,
                    gt_bbox[2] - x1, gt_bbox[3] - y1,
                ]

            if vehicle_crop.size == 0:
                continue

            # --- Estágio 2: Detecta a placa dentro do recorte ---
            plate_detections = plate_detector.detect(vehicle_crop)

            if plate_detections:
                best_plate = max(plate_detections, key=lambda p: p['conf'])
                pred_bbox  = best_plate['bbox']
                px1, py1, px2, py2 = map(int, pred_bbox)
                px1, py1 = max(0, px1), max(0, py1)
                plate_crop = vehicle_crop[py1:py2, px1:px2]
            else:
                # Fallback: usa bbox GT da placa para isolar contribuição do OCR
                pred_bbox  = None
                gx1, gy1, gx2, gy2 = [max(0, int(c)) for c in gt_bbox_crop]
                plate_crop = vehicle_crop[gy1:gy2, gx1:gx2]

            # --- Estágio 3: OCR ---
            pred_text = ''
            if plate_crop is not None and plate_crop.size > 0:
                pred_text = ocr.read_plate(plate_crop) or ''

            # --- Avaliação ---
            metrics = evaluate_detection(pred_bbox, gt_bbox_crop, pred_text, gt_text)

            all_results.append({
                'model':         det_name,
                'track':         img_path.parent.name,   # ex: track0091
                'image':         img_path.name,
                'gt_plate':      gt_text,
                'pred_plate':    pred_text,
                'vehicle_found': len(vehicles) > 0,
                'iou':           round(metrics['iou'], 4),
                'cer':           round(metrics['cer'], 4),
                'exact_match':   metrics['exact_match'],
                'plate_found':   metrics['plate_found'],
            })

    # --- Salva CSV ---
    df = pd.DataFrame(all_results)
    csv_path = OUTPUT_DIR / 'alpr_benchmark.csv'
    df.to_csv(csv_path, index=False)
    print(f"\nResultados salvos em: {csv_path}")

    # --- Gera Relatório ---
    _generate_report(df)

    return df


def _generate_report(df: pd.DataFrame):
    print("\n" + "=" * 60)
    print("  RELATÓRIO FINAL — BENCHMARK UFPR-ALPR")
    print("=" * 60)

    # --- Métricas por frame ---
    summary = df.groupby('model').agg(
        Total_Frames=('image', 'count'),
        IoU_Medio=('iou', 'mean'),
        CER_Medio=('cer', 'mean'),
        Acerto_Frame=('exact_match', 'mean'),
        Taxa_Placa_Encontrada=('plate_found', 'mean'),
    ).round(4)

    # --- Métricas por track (votação por maioria) ---
    # Cada track = 1 veículo com a mesma placa em todos os frames.
    # A leitura OCR mais frequente no track é a "resposta" do sistema.
    def track_accuracy(group):
        """Para cada track, elege a leitura mais votada e compara com GT."""
        def best_vote(g):
            non_empty = g['pred_plate'].dropna()
            non_empty = non_empty[non_empty != '']
            if non_empty.empty:
                voted = ''
            else:
                voted = non_empty.mode().iloc[0]  # leitura mais frequente
            gt = g['gt_plate'].iloc[0]
            return is_exact_match(voted, gt)

        return group.groupby('track').apply(best_vote).mean()

    track_acc = df.groupby('model').apply(track_accuracy).rename('Acerto_Track_Voto')
    summary = summary.join(track_acc)

    summary['Acerto_Frame']           = (summary['Acerto_Frame'] * 100).round(1).astype(str) + '%'
    summary['Taxa_Placa_Encontrada']  = (summary['Taxa_Placa_Encontrada'] * 100).round(1).astype(str) + '%'
    summary['Acerto_Track_Voto']      = (summary['Acerto_Track_Voto'] * 100).round(1).astype(str) + '%'

    print(summary.to_string())
    print()
    print("Nota: 'Acerto_Track_Voto' usa votacao por maioria entre os frames do mesmo track.")
    print("      Cada track corresponde a um veiculo especifico do dataset UFPR-ALPR.")

    # --- Gráficos ---
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('Benchmark ALPR — UFPR-ALPR Dataset', fontsize=14, fontweight='bold')

    plot_df = df.groupby('model')[['iou', 'cer']].mean().reset_index()

    sns.barplot(data=plot_df, x='model', y='iou', hue='model', ax=axes[0], palette='viridis', legend=False)
    axes[0].set_title('IoU Médio (maior = melhor)')
    axes[0].set_ylim(0, 1)
    axes[0].set_ylabel('IoU')

    sns.barplot(data=plot_df, x='model', y='cer', hue='model', ax=axes[1], palette='magma', legend=False)
    axes[1].set_title('CER Médio (menor = melhor)')
    axes[1].set_ylim(0, 1)
    axes[1].set_ylabel('CER (Character Error Rate)')

    # Usa acerto por track (votação) no gráfico — métrica mais justa
    track_vote_df = df.groupby('model').apply(
        lambda g: g.groupby('track').apply(
            lambda t: (
                lambda voted, gt: voted == gt
            )(
                (t['pred_plate'][t['pred_plate'] != ''].mode().iloc[0]
                 if not t['pred_plate'][t['pred_plate'] != ''].empty else ''),
                t['gt_plate'].iloc[0]
            )
        ).mean()
    ).reset_index()
    track_vote_df.columns = ['model', 'acerto_track']

    sns.barplot(data=track_vote_df, x='model', y='acerto_track', hue='model', ax=axes[2], palette='rocket', legend=False)
    axes[2].set_title('Acerto por Track — Votação (maior = melhor)')
    axes[2].set_ylim(0, 1)
    axes[2].set_ylabel('Proporção de Veículos Corretos')

    plt.tight_layout()
    chart_path = OUTPUT_DIR / 'alpr_comparativo.png'
    plt.savefig(chart_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Gráficos salvos em: {chart_path}")




In [ ]:

ufpr_samples = load_dataset_split(UFPR_ROOT, split="testing", limit=6)
print(f"Amostras carregadas para visualização: {len(ufpr_samples)}")

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, s in zip(axes.flat, ufpr_samples):
    img = cv2.imread(str(s["image_path"])).copy()
    vis = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    b   = s["annotation"]["plate_bbox"]
    cv2.rectangle(vis, (b[0], b[1]), (b[2], b[3]), (0, 220, 0), 3)
    ax.imshow(vis); ax.axis("off")
    ax.set_title(f"GT Placa: {s['annotation']['plate_text']}")
plt.tight_layout()
plt.show()

## 3. Demonstração Visual do Pipeline ALPR (Estágio por Estágio)

Célula interativa para exibir visualmente cada etapa da pipeline: Detecção do veículo -> Recorte do veículo -> Detecção da Placa -> Pré-processamento digital -> Leitura OCR final.

In [ ]:

# Inicializa os modelos para demonstração
v_det = YOLODetector("yolov8n.pt")
p_det = PlateDetector()
ocr = OCREngine(engine_type="easyocr")

# Carrega uma amostra do dataset
sample = ufpr_samples[0]
img = cv2.imread(str(sample['image_path']))
h_img, w_img = img.shape[:2]

# Estágio 1: Detecção do Veículo
vehicles = v_det.detect(img)
vehicle_box = [0, 0, w_img, h_img]
for d in vehicles:
    if d['class'] in ['car', 'truck', 'bus', 'motorcycle']:
        vehicle_box = list(map(int, d['bbox']))
        break
vx1, vy1, vx2, vy2 = vehicle_box
veh_crop = img[vy1:vy2, vx1:vx2]

# Estágio 2: Detecção de Placa no Recorte
plates = p_det.detect(veh_crop)
plate_box = [0, 0, veh_crop.shape[1], veh_crop.shape[0]]
if plates:
    plate_box = list(map(int, plates[0]['bbox']))
px1, py1, px2, py2 = plate_box
plate_crop = veh_crop[py1:py2, px1:px2]

# Estágio 3: OCR e Binarização
plate_preprocessed = ocr.preprocess_plate(plate_crop)
plate_text = ocr.read_plate(plate_crop)

# Plotando a cascata do pipeline
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

vis_orig = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
cv2.rectangle(vis_orig, (vx1, vy1), (vx2, vy2), (255, 0, 0), 4)
axes[0].imshow(vis_orig); axes[0].set_title("1. Detecção de Veículo (Blue Box)"); axes[0].axis("off")

vis_veh = cv2.cvtColor(veh_crop, cv2.COLOR_BGR2RGB)
cv2.rectangle(vis_veh, (px1, py1), (px2, py2), (0, 255, 0), 4)
axes[1].imshow(vis_veh); axes[1].set_title("2. Detecção da Placa (Green Box)"); axes[1].axis("off")

vis_plate = cv2.cvtColor(plate_crop, cv2.COLOR_BGR2RGB)
axes[2].imshow(vis_plate); axes[2].set_title("3. Recorte da Placa (Raw)"); axes[2].axis("off")

axes[3].imshow(plate_preprocessed, cmap="gray")
axes[3].set_title(f"4. Binarização + OCR: {plate_text}\n(GT Real: {sample['annotation']['plate_text']})", weight="bold")
axes[3].axis("off")

plt.tight_layout()
plt.show()

## 4. Execução do Benchmark ALPR Completo

In [ ]:

print(f"Iniciando execução do Benchmark ALPR (Limite de imagens: {LIMIT})...")
df_alpr = run_benchmark(split="testing", limit=LIMIT)
df_alpr.to_csv(RESULTS_DIR / "alpr_benchmark.csv", index=False)
display(df_alpr.head())

## 5. Análise Estatística e Visualização de Métricas

Criação dos gráficos comparativos de IoU Médio (geometria da placa), CER Médio (erro do OCR) e taxa de leitura exata por frame para inclusão direta no TCC.

In [ ]:
# Agrupamento estatístico dos resultados por detector de veículo
metrics_summary = df_alpr.groupby('model').agg(
    IoU_Medio=('iou', 'mean'),
    CER_Medio=('cer', 'mean'),
    Exact_Match_Frame=('exact_match', 'mean'),
    Taxa_Placa_Encontrada=('plate_found', 'mean')
).reset_index()

# Configuração estética e plotagem dos gráficos
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Comparativo de Desempenho do Pipeline ALPR por Modelo", fontsize=14, fontweight="bold", y=1.02)

sns.barplot(data=metrics_summary, x="model", y="IoU_Medio", palette="viridis", ax=axes[0], edgecolor="black")
axes[0].set_title("IoU Médio da Placa (Maior = Melhor)")
axes[0].set_ylim(0, 1.0)
axes[0].set_ylabel("IoU")

sns.barplot(data=metrics_summary, x="model", y="CER_Medio", palette="magma", ax=axes[1], edgecolor="black")
axes[1].set_title("Taxa de Erro do OCR (CER - Menor = Melhor)")
axes[1].set_ylim(0, 1.0)
axes[1].set_ylabel("CER")

sns.barplot(data=metrics_summary, x="model", y="Exact_Match_Frame", palette="rocket", ax=axes[2], edgecolor="black")
axes[2].set_title("Acurácia Exata da Placa por Frame")
axes[2].set_ylim(0, 1.0)
axes[2].set_ylabel("Exact Match")

plt.tight_layout()
plt.show()

## 6. Geração de Tabelas LaTeX para a Monografia

Esta célula gera de forma automatizada o código de tabela no formato LaTeX contendo as métricas de frame e a acurácia no nível de veículo (*track*) calculada via votação por maioria.

In [ ]:
def calcular_acerto_track(group):
    def best_vote(g):
        non_empty = g['pred_plate'].dropna()
        non_empty = non_empty[non_empty != '']
        if non_empty.empty:
            voted = ''
        else:
            voted = non_empty.mode().iloc[0]
        gt = g['gt_plate'].iloc[0]
        return voted == gt
    return group.groupby('track').apply(best_vote).mean()

track_acc = df_alpr.groupby('model').apply(calcular_acerto_track).reset_index()
track_acc.columns = ['model', 'Exact_Match_Track']

# Combina métricas de frame e de track
latex_df = pd.merge(metrics_summary, track_acc, on='model')

# Formata colunas
latex_df['Exact_Match_Frame'] = (latex_df['Exact_Match_Frame'] * 100).round(2).astype(str) + '%'
latex_df['Exact_Match_Track'] = (latex_df['Exact_Match_Track'] * 100).round(2).astype(str) + '%'
latex_df['Taxa_Placa_Encontrada'] = (latex_df['Taxa_Placa_Encontrada'] * 100).round(2).astype(str) + '%'
latex_df['IoU_Medio'] = latex_df['IoU_Medio'].round(4)
latex_df['CER_Medio'] = latex_df['CER_Medio'].round(4)

latex_df.columns = ['Algoritmo', 'IoU Médio', 'CER Médio', 'Acurácia (Frame)', 'Localização Placa', 'Acurácia (Track/Votação)']

print('\n' + '='*40)
print('CÓDIGO LATEX AUTOMÁTICO PARA A MONOGRAFIA')
print('='*40)
print(latex_df.to_latex(index=False, caption='Tabela comparativa geral das métricas do pipeline ALPR completo no dataset UFPR-ALPR', label='tab:alpr_geral'))